# V20 · Objekte und CAD-Automatisierung: Klassen, parametrische Bauteile und STL — Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 70 Min). Dieses Notebook ist die **Vertiefung** und läuft am besten im **Codespace** oder in Jupyter, weil du hier **Dateien schreibst** (STL) und sie in einem Programm außerhalb des Browsers öffnest. Es nutzt Python aus V01 bis V19 und neu `class`, `__init__`, `self`, Attribute und Methoden.

**Der rote Faden:** ein Bauteil ist im CAD ein **Satz Parameter plus Regeln**. Wer das als Klasse schreibt, erzeugt jede Variante mit einer Zeile, prüft die Maße automatisch und exportiert das Ergebnis als STL-Datei für 3D-Druck oder Vorschau. Zuerst baust du kleine Klassen ohne CAD (Welle, Rohr, Werkzeug, Flansch-Baureihe), dann nutzt du das Hilfsmodul `bauteil3d` für das Dreiecksnetz und schreibst selbst eine STL-Datei.

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Erst die Klasse lesen: Welche Attribute, welche Methoden, was ist `self`? (2) Vorhersagen, was herauskommt. (3) Ergänzen oder ausführen. (4) Die Prüfzelle laufen lassen. Die Prüfzellen sind keine Musterlösungen, sie zeigen nur, ob dein Ergebnis stimmt.

**Stufen:** Aufwärmen (1.x), Anwenden (2.x), Knobeln und CAD (3.x). Lösungen gibt es hier nicht. Tipps sind eingeklappt.

In [ ]:
# Werkzeug-Zelle für die Selbsttests: einfach einmal ausführen, nicht verändern.
def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")


def pruefe_nah(ist, soll, text, toleranz=1e-6):
    # Für Kommazahlen: "gleich" heißt hier, die Abweichung ist winzig (Rundungsfehler!).
    if isinstance(ist, (int, float)) and abs(ist - soll) <= toleranz:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")


def pruefe_ja(bedingung, text):
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)


print("Bereit.")

## Das Hilfsmodul `bauteil3d` besorgen

Das Modul `bauteil3d.py` erzeugt aus Maßen ein **Dreiecksnetz** (Mesh). Du musst seine Algorithmen nicht verstehen, du **benutzt** es aus deiner Klasse heraus. Die nächste Zelle lädt die Datei von der Kurs-Website in den aktuellen Ordner, falls sie dort noch nicht liegt.

**Falls du offline bist:** Die Datei liegt im Kurs-Repository unter `site/assets/py/bauteil3d.py`. Kopiere sie in den Ordner dieses Notebooks (im Codespace liegt sie schon im Repository, die Zelle findet sie dann selbst). Die Aufgaben 1.x und 2.x laufen auch **ohne** `bauteil3d`.

In [ ]:
import os
import sys
import urllib.request

URL = "https://nice-wave-051f81f03.3.azurestaticapps.net/assets/py/bauteil3d.py"

# Im Codespace liegt die Datei schon im Repository: dann nur den Suchpfad ergänzen.
if os.path.exists("site/assets/py/bauteil3d.py"):
    sys.path.append("site/assets/py")

if not os.path.exists("bauteil3d.py") and not os.path.exists("site/assets/py/bauteil3d.py"):
    try:
        urllib.request.urlretrieve(URL, "bauteil3d.py")
        print("bauteil3d.py heruntergeladen.")
    except Exception as fehler:
        print("Download nicht möglich:", fehler)
        print("Kopiere site/assets/py/bauteil3d.py von Hand in diesen Ordner (siehe Text oben).")

try:
    import bauteil3d
    print("bauteil3d bereit. Funktionen:", [n for n in dir(bauteil3d) if not n.startswith("_") and callable(getattr(bauteil3d, n))])
except ImportError:
    bauteil3d = None
    print("bauteil3d fehlt: Aufgaben 3.x brauchen es. 1.x und 2.x gehen trotzdem.")

## Stufe 1 — Aufwärmen

### 1.1 Klasse lesen und vorhersagen
Lies die Klasse. Schreibe **vor dem Ausführen** als Kommentar in die erste Zeile, was die beiden `print` ausgeben. Achte darauf, welches Objekt `self` ist, wenn `streckung` läuft, und dass `b.l` geändert wird.

<details><summary>Tipp anzeigen</summary>

Tipp: `a` und `b` sind zwei **getrennte** Objekte mit eigenen Attributen. `a.streckung()` rechnet mit `a.l / a.d` = 150 / 20, `b.streckung()` mit `b.l / b.d` = 300 / 40. Dass beide gleich sind, ist Zufall der Zahlen, nicht der Klasse.
</details>

In [ ]:
# Meine Vorhersage: ...
class Welle:
    def __init__(self, d, l):
        self.d = d
        self.l = l

    def streckung(self):
        return self.l / self.d


a = Welle(20, 150)
b = Welle(40, 150)
b.l = 300
print(a.streckung())
print(b.streckung())

### 1.2 Eine Klasse schreiben: Welle
Vervollständige die Klasse `Welle` (Maße in **mm**). `__init__` speichert `d` und `l` als Attribute. `volumen()` gibt das Volumen des Zylinders in **mm³** zurück. `masse(dichte=7.85)` gibt die Masse in **kg** zurück, die Dichte steht in g/cm³ (Stahl 7,85). Einheitenkette: mm³ geteilt durch 1000 sind cm³, mal Dichte sind g, geteilt durch 1000 sind kg.

<details><summary>Tipp anzeigen</summary>

Tipp: `self.d = d` in `__init__`. Volumen: `math.pi / 4 * self.d ** 2 * self.l`. Masse: `self.volumen() / 1000 * dichte / 1000`. In jeder Methode steht `self` als erster Parameter, und die Attribute heißen `self.d`, nicht `d`.
</details>

In [ ]:
import math


class Welle:
    def __init__(self, d, l):
        # TODO: beide Maße als Attribute speichern
        pass

    def volumen(self):
        # TODO: Volumen in mm³ (Zylinder)
        return None

    def masse(self, dichte=7.85):
        # TODO: Masse in kg (mm³ -> cm³ -> g -> kg)
        return None


w = Welle(20, 150)
print(w.volumen())
print(w.masse())

In [ ]:
pruefe(getattr(w, "d", None), 20, "Attribut d gespeichert")
pruefe(getattr(w, "l", None), 150, "Attribut l gespeichert")
pruefe_nah(w.volumen(), math.pi / 4 * 20 ** 2 * 150, "Volumen in mm³")
pruefe_nah(w.masse(), 0.36994, "Masse Stahl in kg", 1e-4)
pruefe_nah(w.masse(2.70), 0.12723, "Masse Aluminium (Dichte 2,70) in kg", 1e-4)
w2 = Welle(40, 100)
pruefe_nah(w2.volumen(), math.pi / 4 * 40 ** 2 * 100, "zweites Objekt hat eigene Maße")

### 1.3 Zwei Namen, ein Objekt
Sage erst vorher, was die drei Zeilen ausgeben. Dann ausführen. Der Unterschied zwischen `Welle(20, 150)` (ein **neues** Objekt) und `w2 = w1` (ein **zweiter Name** für dasselbe Objekt) kennst du von Listen aus V08.

In [ ]:
# Meine Vorhersage: ...
class Welle:
    def __init__(self, d, l):
        self.d = d
        self.l = l


w1 = Welle(20, 150)
w2 = w1
w3 = Welle(20, 150)
w2.d = 99
print(w1.d, w2.d, w3.d)
print(w1 is w2, w1 is w3)

## Stufe 2 — Anwenden

### 2.1 Rohr mit Prüfung
Schreibe die Klasse `Rohr(d_aussen, d_innen, l)` (Maße in mm). `ist_gueltig()` gibt `True` zurück, wenn `0 < d_innen < d_aussen` und `l > 0` gilt, sonst `False`. `volumen()` ist der Kreisring mal Länge in mm³. `masse(dichte=7.85)` gibt kg zurück. Eine **Methode darf eine andere Methode aufrufen**: `self.volumen()`.

<details><summary>Tipp anzeigen</summary>

Tipp: Kreisring: `math.pi / 4 * (self.d_aussen ** 2 - self.d_innen ** 2) * self.l`. Prüfung: `return 0 < self.d_innen < self.d_aussen and self.l > 0`. In `masse` rufst du `self.volumen()` mit Klammern auf.
</details>

In [ ]:
import math


class Rohr:
    # TODO: __init__, ist_gueltig, volumen, masse
    pass


r = Rohr(42, 36, 1000)
print(r.ist_gueltig(), r.volumen(), r.masse())

In [ ]:
pruefe(r.ist_gueltig(), True, "Rohr 42/36 ist gültig")
pruefe(Rohr(36, 42, 1000).ist_gueltig(), False, "Innen größer als außen ist ungültig")
pruefe(Rohr(42, 36, 0).ist_gueltig(), False, "Länge 0 ist ungültig")
pruefe_nah(r.volumen(), 367566.3, "Volumen in mm³", 0.1)
pruefe_nah(r.masse(), 2.8854, "Masse Stahl in kg", 1e-3)
pruefe_nah(r.masse(2.70), 0.9924, "Masse Aluminium in kg", 1e-3)

### 2.2 Eine Methode verändert das Objekt
Ein Werkzeug hat eine Standzeit. Schreibe `Werkzeug(standzeit_min)`: Attribut `rest` (Rest-Standzeit in min, am Anfang gleich `standzeit_min`). `benutze(minuten)` verringert `rest` um `minuten`, aber `rest` darf nicht unter 0 fallen. `ist_verschlissen()` gibt `True` zurück, sobald `rest` 0 erreicht hat. Diese Methoden **geben nichts zurück** (`benutze`) oder einen Wahrheitswert (`ist_verschlissen`): Der Zustand steckt im Objekt.

<details><summary>Tipp anzeigen</summary>

Tipp: `self.rest = self.rest - minuten`, danach `if self.rest < 0: self.rest = 0`. Oder in einer Zeile mit `max(0, ...)`. Zwei Werkzeuge sind zwei Objekte: `t1.benutze(10)` ändert `t2` nicht.
</details>

In [ ]:
class Werkzeug:
    # TODO: __init__, benutze, ist_verschlissen
    pass


t1 = Werkzeug(60)
t2 = Werkzeug(60)
t1.benutze(25)
print(t1.rest, t2.rest, t1.ist_verschlissen())

In [ ]:
pruefe(t1.rest, 35, "nach 25 min bleiben 35 min")
pruefe(t2.rest, 60, "das zweite Werkzeug ist unberührt")
pruefe(t1.ist_verschlissen(), False, "noch nicht verschlissen")
t1.benutze(40)
pruefe(t1.rest, 0, "Rest-Standzeit fällt nicht unter 0")
pruefe(t1.ist_verschlissen(), True, "jetzt verschlissen")

### 2.3 Eine Baureihe als Liste von Objekten
Die Konstrukteurin liefert eine Baureihe von Flanschen. Die Klasse `Flansch` ist fertig (lies sie!). Ergänze darunter: (a) `gueltige`: Liste aller **gültigen** Flansche, (b) `ungueltig`: Anzahl der ungültigen, (c) `schwerster`: der gültige Flansch mit der größten Masse (das Objekt selbst, nicht die Zahl). Eine Methode wie `f.masse()` rufst du in einer Schleife oder List Comprehension für jedes Objekt auf.

<details><summary>Tipp anzeigen</summary>

Tipp: `gueltige = [f for f in reihe if f.ist_gueltig()]`. Anzahl ungültig: `len(reihe) - len(gueltige)`. Das schwerste Objekt: `max(gueltige, key=lambda f: f.masse())` oder mit einer Schleife, die sich das bisher schwerste merkt.
</details>

In [ ]:
import math


class Flansch:
    def __init__(self, name, d_aussen, d_innen, dicke, lochkreis, n, d_bohrung):
        self.name = name
        self.d_aussen = d_aussen
        self.d_innen = d_innen
        self.dicke = dicke
        self.lochkreis = lochkreis
        self.n = n
        self.d_bohrung = d_bohrung

    def ist_gueltig(self):
        rand_aussen = (self.lochkreis + self.d_bohrung) / 2 < self.d_aussen / 2
        rand_innen = (self.lochkreis - self.d_bohrung) / 2 > self.d_innen / 2
        return rand_aussen and rand_innen

    def volumen(self):
        ring = math.pi / 4 * (self.d_aussen ** 2 - self.d_innen ** 2) * self.dicke
        bohrungen = self.n * math.pi / 4 * self.d_bohrung ** 2 * self.dicke
        return ring - bohrungen

    def masse(self, dichte=7.85):
        return self.volumen() / 1000 * dichte / 1000


reihe = [
    Flansch("DN25", 115, 25, 16, 85, 4, 14),
    Flansch("DN40", 150, 40, 18, 110, 4, 18),
    Flansch("DN50", 165, 50, 20, 125, 4, 18),
    Flansch("DN65", 185, 65, 20, 145, 8, 18),
    Flansch("DN80", 200, 80, 22, 160, 8, 18),
    Flansch("DN100", 220, 100, 22, 210, 8, 22),    # Tippfehler im Datenblatt: Lochkreis 210?
]

# TODO: gueltige, ungueltig, schwerster
gueltige = None
ungueltig = None
schwerster = None
print(ungueltig, schwerster.name if schwerster else None)

In [ ]:
pruefe(len(gueltige) if gueltige else None, 5, "fünf gültige Flansche")
pruefe(ungueltig, 1, "ein ungültiger Flansch")
pruefe(getattr(schwerster, "name", None), "DN80", "schwerster gültiger Flansch")
pruefe_nah(getattr(schwerster, "masse", lambda: None)(), 4.2059, "dessen Masse in kg", 1e-3)
pruefe(all(f.ist_gueltig() for f in gueltige) if gueltige else False, True, "die Liste enthält nur gültige")

## Stufe 3 — Knobeln und CAD

### 3.1 Von der Klasse zum 3D-Netz
Der Flansch bekommt eine Methode `mesh()`, die das Dreiecksnetz liefert. Du **delegierst** die Geometrie an `bauteil3d.flansch(d_aussen, d_innen, dicke, lochkreis, n, d_bohrung)`. Das Netz ist ein Dictionary mit den Schlüsseln `"vertices"` (Eckpunkte) und `"faces"` (Dreiecke aus je drei Punkt-Nummern). Prüfe danach: Das Volumen des Netzes (`bauteil3d.volumen(netz)`) weicht nur wenig von deiner Formel ab, weil Kreise im Netz aus Geraden bestehen.

<details><summary>Tipp anzeigen</summary>

Tipp: `return bauteil3d.flansch(self.d_aussen, self.d_innen, self.dicke, self.lochkreis, self.n, self.d_bohrung)`. Wenn `bauteil3d` fehlt, lies oben den Abschnitt „Das Hilfsmodul besorgen“.
</details>

In [ ]:
import math
import bauteil3d


class Flansch:
    def __init__(self, d_aussen, d_innen, dicke, lochkreis, n, d_bohrung):
        self.d_aussen = d_aussen
        self.d_innen = d_innen
        self.dicke = dicke
        self.lochkreis = lochkreis
        self.n = n
        self.d_bohrung = d_bohrung

    def volumen(self):
        ring = math.pi / 4 * (self.d_aussen ** 2 - self.d_innen ** 2) * self.dicke
        bohrungen = self.n * math.pi / 4 * self.d_bohrung ** 2 * self.dicke
        return ring - bohrungen

    def mesh(self):
        # TODO: Netz über bauteil3d.flansch erzeugen und zurückgeben
        return None


f = Flansch(120, 40, 15, 90, 6, 11)
netz = f.mesh()
print(type(netz))

In [ ]:
pruefe_ja(isinstance(netz, dict) and "faces" in netz, "mesh() liefert ein Netz (Dictionary mit faces)")
if isinstance(netz, dict) and "faces" in netz:
    print("Eckpunkte:", len(netz["vertices"]), " Dreiecke:", len(netz["faces"]))
    v_netz = bauteil3d.volumen(netz)
    abweichung = abs(v_netz - f.volumen()) / f.volumen()
    print(f"Volumen Formel {f.volumen() / 1000:.1f} cm³, Netz {v_netz / 1000:.1f} cm³, Abweichung {abweichung * 100:.2f} %")
    pruefe_ja(abweichung < 0.01, "Netz-Volumen weicht weniger als 1 % von der Formel ab")
    pruefe_ja(len(netz["faces"]) > 100, "viele kleine Dreiecke")

### 3.2 Eine STL-Datei schreiben
**STL** ist das Standardformat für 3D-Druck und einfache Vorschau: eine Liste von Dreiecken. In der ASCII-Variante besteht jedes Dreieck aus einer **Normalen** (Vektor senkrecht zur Fläche, zeigt nach außen) und drei Eckpunkten:

```
solid Flansch
  facet normal 0 0 1
    outer loop
      vertex 0 0 15
      vertex 10 0 15
      vertex 0 10 15
    endloop
  endfacet
endsolid Flansch
```

Die Funktion `stl_schreiben(mesh, dateiname)` ist fast fertig. Es fehlt die **Berechnung der Normalen** aus den drei Eckpunkten `p1`, `p2`, `p3`: Bilde die Kantenvektoren `u = p2 - p1` und `v = p3 - p1`, daraus das Kreuzprodukt `n = u × v` und teile es durch seine Länge (damit der Vektor die Länge 1 hat). Bei einem Dreieck ohne Fläche (Länge 0) nimmst du `(0, 0, 0)`.

Das Format im Detail musst du nicht auswendig kennen. Der Zweck der Aufgabe: Du siehst, dass ein 3D-Modell am Ende **nur eine Textdatei mit Zahlen** ist, die dein Programm erzeugt hat.

<details><summary>Tipp anzeigen</summary>

Tipp: Das Kreuzprodukt hat die Komponenten `nx = uy*vz - uz*vy`, `ny = uz*vx - ux*vz`, `nz = ux*vy - uy*vx`. Länge: `math.sqrt(nx**2 + ny**2 + nz**2)`. Teile jede Komponente durch die Länge, falls sie größer als 0 ist.
</details>

In [ ]:
import math


def normale(p1, p2, p3):
    """Einheitsvektor senkrecht zum Dreieck p1-p2-p3 (nach der Rechte-Hand-Regel)."""
    # TODO: Kantenvektoren u, v, Kreuzprodukt, durch die Länge teilen
    return (0.0, 0.0, 0.0)


def stl_schreiben(mesh, dateiname):
    """Schreibt ein Mesh als ASCII-STL und gibt die Zahl der Dreiecke zurück."""
    name = mesh.get("name", "Bauteil")
    pkt = mesh["vertices"]
    with open(dateiname, "w", encoding="utf-8") as datei:
        datei.write(f"solid {name}\n")
        for i, j, k in mesh["faces"]:
            n = normale(pkt[i], pkt[j], pkt[k])
            datei.write(f"  facet normal {n[0]:.6f} {n[1]:.6f} {n[2]:.6f}\n")
            datei.write("    outer loop\n")
            for index in (i, j, k):
                x, y, z = pkt[index]
                datei.write(f"      vertex {x:.4f} {y:.4f} {z:.4f}\n")
            datei.write("    endloop\n")
            datei.write("  endfacet\n")
        datei.write(f"endsolid {name}\n")
    return len(mesh["faces"])


anzahl = stl_schreiben(netz, "flansch.stl")
print(anzahl, "Dreiecke in flansch.stl geschrieben")

In [ ]:
with open("flansch.stl", encoding="utf-8") as datei:
    zeilen = datei.read().splitlines()
facetten = [z for z in zeilen if z.strip().startswith("facet normal")]
laengen = []
for z in facetten:
    nx, ny, nz = (float(t) for t in z.split()[2:5])
    laengen.append(math.sqrt(nx ** 2 + ny ** 2 + nz ** 2))

pruefe(zeilen[0], "solid Flansch", "Datei beginnt mit solid und dem Namen")
pruefe(zeilen[-1], "endsolid Flansch", "Datei endet mit endsolid")
pruefe(len(facetten), len(netz["faces"]), "ein facet pro Dreieck")
pruefe_ja(all(abs(l - 1) < 1e-4 for l in laengen), "alle Normalen haben die Länge 1")
oben = [z for z in facetten if z.split()[2:5] == ["0.000000", "0.000000", "1.000000"]]
pruefe_ja(len(oben) > 20, "die Deckfläche (z = 15) hat die Normale (0, 0, 1)")
print()
print("Die Datei flansch.stl liegt jetzt in diesem Ordner. Öffne sie in einem STL-Viewer oder Slicer")
print("(zum Beispiel dem 3D-Viewer von Windows, PrusaSlicer, Cura oder FreeCAD) und drehe das Bauteil.")

### 3.3 Stufenwelle: Liste von Tupeln als Parameter
Eine Stufenwelle besteht aus mehreren Absätzen `(Durchmesser, Länge)`. Schreibe die Klasse `Stufenwelle(absaetze)`: `gesamtlaenge()` (Summe der Längen in mm), `volumen()` (Summe der Zylindervolumen in mm³, **ohne** `bauteil3d`) und `mesh()` über `bauteil3d.welle(self.absaetze)`. Danach schreibst du die STL-Datei mit der Funktion aus 3.2.

<details><summary>Tipp anzeigen</summary>

Tipp: In `volumen` läufst du mit `for d, l in self.absaetze:` über die Tupel (Tupel entpacken, V14) und addierst `math.pi / 4 * d ** 2 * l`. Die Gesamtlänge ist `sum(l for d, l in self.absaetze)`.
</details>

In [ ]:
import math


class Stufenwelle:
    def __init__(self, absaetze):
        self.absaetze = absaetze

    # TODO: gesamtlaenge, volumen, mesh


sw = Stufenwelle([(20, 40), (30, 60), (25, 30)])
print(sw.gesamtlaenge(), sw.volumen())

In [ ]:
pruefe(sw.gesamtlaenge(), 130, "Gesamtlänge 130 mm")
pruefe_nah(sw.volumen(), 69704.1, "Volumen in mm³", 0.5)
wnetz = sw.mesh()
pruefe_ja(isinstance(wnetz, dict) and "faces" in wnetz, "mesh() liefert ein Netz")
if isinstance(wnetz, dict) and "faces" in wnetz:
    abw = abs(bauteil3d.volumen(wnetz) - sw.volumen()) / sw.volumen()
    pruefe_ja(abw < 0.01, f"Netz-Volumen weicht {abw * 100:.2f} % ab (unter 1 %)")
    print(stl_schreiben(wnetz, "stufenwelle.stl"), "Dreiecke in stufenwelle.stl")

### 3.4 Baugruppe aus drei Hülsen (frei)
Schreibe die Klasse `Distanzhuelse(d_aussen, d_innen, l)` mit `mesh()` über `bauteil3d.ring(...)`. Stelle drei Hülsen mit unterschiedlicher Länge nebeneinander: `bauteil3d.verschiebe(netz, dx, dy, dz)` schiebt ein Netz, `bauteil3d.vereinige(netz1, netz2, netz3, name="Baugruppe")` hängt sie zusammen (ohne sie zu verschmelzen). Schreibe die Baugruppe als `huelsen.stl` und öffne sie im Viewer. Die Prüfzelle prüft nur, ob die Zahl der Dreiecke zu drei Hülsen passt.

In [ ]:
# Dein Code: Klasse Distanzhuelse, drei Objekte, verschieben, vereinigen, STL schreiben.
# Die Variable baugruppe soll am Ende das vereinigte Netz enthalten, huelse1 das erste Objekt.
huelse1 = None
baugruppe = None

In [ ]:
if baugruppe is None or huelse1 is None:
    print("Noch nicht bearbeitet.")
else:
    einzeln = len(huelse1.mesh()["faces"])
    pruefe(len(baugruppe["faces"]), 3 * einzeln, "Baugruppe hat dreimal so viele Dreiecke wie eine Hülse")

## Ausblick: FreeCAD und CadQuery (freiwillig, nicht ausführbar hier)
Was du eben mit `bauteil3d` gemacht hast, machen CAD-Programme mit echter Geometrie: Sie speichern ein Bauteil als **Feature-Baum** (Skizze, Extrusion, Bohrung, Muster) mit Parametern, und ihre **Skript-Schnittstelle** (API) lässt dich diese Schritte aus Python steuern. Zwei Wege:

- **FreeCAD** hat eine eingebaute Python-Konsole und Makros. Du schreibst dieselbe Klasse mit Maßen und Methoden, die Methoden rufen FreeCAD-Befehle statt `bauteil3d`.
- **CadQuery** ist eine Python-Bibliothek, die nur mit Code ein Volumenmodell aufbaut. Der Flansch von oben sieht ungefähr so aus (**nicht** ausführen: Du brauchst eine Installation von CadQuery, z. B. im Codespace mit `pip install cadquery`):

```python
# pip install cadquery   (läuft nicht im Browser)
import cadquery as cq

class Flansch:
    def __init__(self, d_aussen, d_innen, dicke, lochkreis, n, d_bohrung):
        self.d_aussen, self.d_innen, self.dicke = d_aussen, d_innen, dicke
        self.lochkreis, self.n, self.d_bohrung = lochkreis, n, d_bohrung

    def modell(self):
        return (cq.Workplane("XY")
                .circle(self.d_aussen / 2).circle(self.d_innen / 2).extrude(self.dicke)
                .faces(">Z").workplane()
                .polarArray(self.lochkreis / 2, 0, 360, self.n)
                .hole(self.d_bohrung))

f = Flansch(120, 40, 15, 90, 6, 11)
cq.exporters.export(f.modell(), "flansch.step")   # STEP: exakte Geometrie für die Fertigung
cq.exporters.export(f.modell(), "flansch.stl")    # STL: Dreiecksnetz für 3D-Druck
```

Du erkennst die Bausteine wieder: eine Klasse mit Maßen, eine Methode, die das Bauteil liefert, eine Zeile pro Variante. **STL** reicht für 3D-Druck und Vorschau (Näherung aus Dreiecken). **STEP** enthält die exakte Geometrie und ist das Austauschformat für Konstruktion und Fertigung. Wie die CAD-API im Detail heißt, **delegierst** du (an die Dokumentation oder an KI). Dein Anteil: die Klasse lesen, die Maße und Regeln prüfen (Bohrung über den Rand?) und das Ergebnis im Viewer ansehen.

## Selbsttest: Kannst du es im Kopf?
Beantworte erst auf Papier, dann prüfe im Notebook.

1. Was ist der Unterschied zwischen einer **Klasse** und einem **Objekt**? Nenne ein Beispiel aus dem Maschinenbau.
2. Gegeben `w1 = Welle(20, 150)`, `w2 = w1`, `w2.l = 80`. Was ist `w1.l`? Wie viele Objekte gibt es?
3. Welche der Aufrufe `w.volumen`, `w.volumen()`, `Welle.volumen()` liefert die Zahl, und was geben die anderen aus bzw. was passiert?
4. In `class Welle` steht `def umfang(): return 3.14 * self.d`. Welche Fehlermeldung bekommst du bei `w.umfang()`, und was ist die Korrektur?
5. Eine Welle hat d = 30 mm und l = 200 mm. Wie viel kg wiegt sie aus Stahl (7,85 g/cm³)? Rechne mm³, cm³, g, kg einzeln.

In [ ]:
import math


class Welle:
    def __init__(self, d, l):
        self.d = d
        self.l = l

    def umfang(self):
        return 3.14 * self.d


w1 = Welle(20, 150)
w2 = w1
w2.l = 80
print(w1.l, w1 is w2)                                  # 2
print(w1.umfang)                                       # 3: ohne Klammern kommt die Methode, nicht die Zahl
print(w1.umfang())                                     # 3: mit Klammern kommt das Ergebnis
volumen_mm3 = math.pi / 4 * 30 ** 2 * 200              # 5
print(round(volumen_mm3 / 1000, 1), "cm³ ->", round(volumen_mm3 / 1000 * 7.85 / 1000, 2), "kg")

In [ ]:
class Falsch:
    def __init__(self, d):
        self.d = d

    def umfang():                                      # 4: self fehlt in der Definition
        return 3.14 * self.d


try:
    Falsch(20).umfang()
except TypeError as fehler:
    print("TypeError:", fehler)